# Validación contra las 743 anotaciones reales (KPCL0034)

**Objetivo de este notebook:** responder "¿esto está funcionando?" con
evidencia, no solo con silhouette. Silhouette mide qué tan separados quedan
los clusters entre sí — **no dice si esos clusters corresponden a
alimentación/servido/ruido de verdad**.

**Método (excepción acotada, igual que antes):** se usan las 743
anotaciones reales (`t_inicio`, `t_fin`, `categoria`) **solo como validación
por solapamiento de tiempo** — nunca entran a la detección, al clustering, ni
a ninguna feature. Es evaluar el resultado después de construido, no
entrenar con eso.

Dos preguntas concretas:
1. **Cobertura**: de los eventos reales ya anotados, ¿cuántos coinciden en
   el tiempo con alguno de nuestros candidatos (τ=180s)?
2. **Pureza de cluster**: de los candidatos que sí coinciden con un evento
   real, ¿el cluster de KMeans corresponde a la categoría real, o los
   clusters mezclan categorías?


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
ANOTACIONES_CSV = NOTEBOOK_DIR.parent / "Ciclo_Alpha_v2" / "fase_0_ruido" / "data" / "anotaciones_av2.csv"
CANDIDATOS_CSV = NOTEBOOK_DIR / "data" / "candidatos_clusters_duracion.csv"

anot = pd.read_csv(ANOTACIONES_CSV)
anot["t_inicio"] = pd.to_datetime(anot["t_inicio"], format="ISO8601", utc=True)
anot["t_fin"] = pd.to_datetime(anot["t_fin"], format="ISO8601", utc=True)

cand = pd.read_csv(CANDIDATOS_CSV)
cand["ts_inicio"] = pd.to_datetime(cand["ts_inicio"], format="ISO8601", utc=True)
cand["ts_fin"] = pd.to_datetime(cand["ts_fin"], format="ISO8601", utc=True)
cand = cand[cand["device_code"] == "KPCL0034"].reset_index(drop=True)

print(f"Anotaciones reales: {len(anot):,}  ({anot['t_inicio'].min():%Y-%m-%d} a {anot['t_fin'].max():%Y-%m-%d})")
print(f"Candidatos (tau=180s): {len(cand):,}  ({cand['ts_inicio'].min():%Y-%m-%d} a {cand['ts_inicio'].max():%Y-%m-%d})")


## Cobertura: ¿cuántos eventos reales caen dentro del rango de fechas donde tenemos candidatos?

Primero acotar ambos al período que se solapa en el tiempo — no tiene
sentido pedirle cobertura sobre fechas donde uno de los dos no tiene datos.


In [ ]:
_inicio_comun = max(anot["t_inicio"].min(), cand["ts_inicio"].min())
_fin_comun = min(anot["t_fin"].max(), cand["ts_inicio"].max())
print(f"Periodo comun: {_inicio_comun:%Y-%m-%d} a {_fin_comun:%Y-%m-%d}")

anot_comun = anot[(anot["t_inicio"] >= _inicio_comun) & (anot["t_fin"] <= _fin_comun)].copy()
print(f"Anotaciones dentro del periodo comun: {len(anot_comun):,} de {len(anot):,}")
print(anot_comun["categoria"].value_counts())


## Solapamiento por tiempo (interval overlap), no por texto ni por features

Un candidato "cubre" una anotación si sus rangos `[ts_inicio, ts_fin]` se
solapan en el tiempo (cualquier solapamiento, no exige coincidencia exacta
— los bordes van a diferir un poco entre el corte automático y el humano).


In [ ]:
def hay_solapamiento(a_ini, a_fin, c_ini, c_fin):
    return (a_ini <= c_fin) & (a_fin >= c_ini)

filas_cobertura = []
for _, _a in anot_comun.iterrows():
    _solapa = hay_solapamiento(_a["t_inicio"], _a["t_fin"], cand["ts_inicio"], cand["ts_fin"])
    _candidatos_solapados = cand[_solapa]
    filas_cobertura.append({
        "id_anotacion": _a["id_anotacion"], "categoria": _a["categoria"],
        "duracion_anotacion_s": (_a["t_fin"] - _a["t_inicio"]).total_seconds(),
        "n_candidatos_solapados": len(_candidatos_solapados),
        "cluster_kmeans_mas_grande": (
            _candidatos_solapados.loc[_candidatos_solapados["duracion_s"].idxmax(), "cluster_kmeans"]
            if len(_candidatos_solapados) > 0 else np.nan
        ),
    })
cobertura_df = pd.DataFrame(filas_cobertura)

cobertura_df["cubierta"] = cobertura_df["n_candidatos_solapados"] > 0
print("--- Cobertura por categoria ---")
print(cobertura_df.groupby("categoria")["cubierta"].agg(["sum", "count", "mean"]).round(3))


## Pureza de cluster: de lo que SÍ coincide con un evento real, ¿el cluster corresponde a la categoría?


In [ ]:
cubiertas = cobertura_df[cobertura_df["cubierta"]].dropna(subset=["cluster_kmeans_mas_grande"])
print(f"Anotaciones cubiertas con cluster asignado: {len(cubiertas):,}")
print()
tabla_cruzada = pd.crosstab(cubiertas["cluster_kmeans_mas_grande"], cubiertas["categoria"])
print("Cluster KMeans (filas) vs categoria real (columnas):")
print(tabla_cruzada)

print()
print("Composicion de cada cluster, en % de categoria real:")
print((tabla_cruzada.div(tabla_cruzada.sum(axis=1), axis=0) * 100).round(1))


## Candidatos SIN ninguna anotación real cerca (dentro del período común)

No necesariamente son errores — pueden ser eventos reales que nunca se
anotaron (las 743 anotaciones no cubren el 100% de la actividad real).
Vale la pena saber cuántos son para no sobre-interpretar la cobertura.


In [ ]:
cand_comun = cand[(cand["ts_inicio"] >= _inicio_comun) & (cand["ts_inicio"] <= _fin_comun)].copy()
def tiene_anotacion_cerca(c_ini, c_fin):
    return hay_solapamiento(anot_comun["t_inicio"], anot_comun["t_fin"], c_ini, c_fin).any()
cand_comun["tiene_anotacion"] = [
    tiene_anotacion_cerca(row["ts_inicio"], row["ts_fin"]) for _, row in cand_comun.iterrows()
]
_sin_anot = ~cand_comun["tiene_anotacion"]
print(f"Candidatos sin ninguna anotacion real cerca: "
      f"{_sin_anot.sum():,} de {len(cand_comun):,} "
      f"({_sin_anot.mean() * 100:.1f}%)")


## Exportar categoría real por candidato (para verla en la app)

Por cada candidato, si se solapa con una o más anotaciones reales, se toma
la categoría de la que más se solapa en tiempo. Si no se solapa con
ninguna, queda como `"sin_anotacion"` — no es un error, solo no hay
verificación humana para revisarlo contra algo.


In [ ]:
def categoria_real_de(c_ini, c_fin):
    _solapa = hay_solapamiento(anot_comun["t_inicio"], anot_comun["t_fin"], c_ini, c_fin)
    _candidatas = anot_comun[_solapa]
    if len(_candidatas) == 0:
        return "sin_anotacion"
    _solape_s = (
        _candidatas[["t_inicio", "t_fin"]].clip(lower=c_ini, upper=c_fin, axis=0)
        .pipe(lambda d: (d["t_fin"] - d["t_inicio"]).dt.total_seconds())
    )
    return _candidatas.loc[_solape_s.idxmax(), "categoria"]


cand_comun["categoria_real"] = [
    categoria_real_de(row["ts_inicio"], row["ts_fin"]) for _, row in cand_comun.iterrows()
]
print(cand_comun["categoria_real"].value_counts())

CATEGORIA_REAL_CSV = NOTEBOOK_DIR / "data" / "candidatos_categoria_real.csv"
cand_comun[["candidato_id", "categoria_real"]].to_csv(CATEGORIA_REAL_CSV, index=False)
print(f"Exportado: {CATEGORIA_REAL_CSV}")


## Promover veredictos manuales confirmados a `categoria_real`

El modo "Revisar candidatos sin anotación real" de la app (`visualizacion/
app_candidatos.py`) guarda en `data/revision_sin_anotacion.csv` el veredicto
manual sobre candidatos que **no** tenían ninguna anotación real cerca —
revisados a ojo por el usuario, confirmando o corrigiendo lo que el cluster
sugería. Esto **no es una anotación de Ciclo_Alpha_v2** — es una anotación
manual propia, hecha directo sobre nuestros candidatos.

Se promueven a `categoria_real` los veredictos que son una categoría real
confirmada (`alimentacion`/`servido`/`ruido`) — se excluyen `(sin revisar)`
y `no está claro`, que no son una confirmación.


In [ ]:
REVISION_CSV = NOTEBOOK_DIR / "data" / "revision_sin_anotacion.csv"
if REVISION_CSV.exists():
    revision = pd.read_csv(REVISION_CSV)
    _confirmados = revision[revision["veredicto"].isin(["alimentacion", "servido", "ruido"])]
    print(f"Veredictos manuales confirmados: {len(_confirmados):,} de {len(revision):,} revisados")
    print(_confirmados["veredicto"].value_counts())

    categoria_real_df = pd.read_csv(CATEGORIA_REAL_CSV)
    _antes = (categoria_real_df["categoria_real"] == "sin_anotacion").sum()
    categoria_real_df = categoria_real_df.set_index("candidato_id")
    _veredictos_por_id = _confirmados.set_index("candidato_id")["veredicto"]
    categoria_real_df.loc[_veredictos_por_id.index, "categoria_real"] = _veredictos_por_id
    categoria_real_df = categoria_real_df.reset_index()
    _despues = (categoria_real_df["categoria_real"] == "sin_anotacion").sum()

    categoria_real_df.to_csv(CATEGORIA_REAL_CSV, index=False)
    print()
    print(f"'sin_anotacion' antes: {_antes:,} -> despues: {_despues:,} "
          f"({_antes - _despues:,} promovidos a categoria real confirmada)")
    print(f"Actualizado: {CATEGORIA_REAL_CSV}")
else:
    print("Todavia no hay veredictos manuales guardados (revision_sin_anotacion.csv no existe).")


## Cierre

**Resultado real (2026-08-30):**

### Cobertura (¿detectamos los eventos reales?)

| Categoría | Cubiertas | Total | % |
|---|---|---|---|
| Alimentación | 318 | 318 | **100%** |
| Ruido | 289 | 350 | 82.6% |
| Servido | 62 | 75 | 82.7% |

Detectamos el 100% de las alimentaciones reales, y más del 80% de servido
y ruido — con τ=180s, ya no se nos escapan eventos completos.

### Pureza de cluster (¿el cluster corresponde a la categoría real?)

| Cluster KMeans | % alimentación | % ruido | % servido | Lectura |
|---|---|---|---|---|
| 0 (n=425) | 70.8% | 24.2% | 4.9% | dominado por alimentación |
| 1 (n=153) | 11.1% | 88.2% | 0.7% | dominado por ruido |
| 2 (n=91) | 0.0% | 56.0% | 44.0% | mezcla ruido/servido, pero excluye alimentación del todo |

**Primera vez que un cluster corresponde de verdad a una categoría de
dominio** — a diferencia de todos los intentos anteriores (motor viejo,
features propias con τ=1 lectura, shape features), donde el cluster grande
genérico mezclaba todo. El cluster 0 es mayoritariamente alimentación real,
el cluster 1 es mayoritariamente ruido real. Servido todavía no tiene un
cluster propio — queda repartido entre el 0 y el 2, mezclado con ruido en
el 2.

### Lo que falta

- **149 de 701 candidatos (21.3%)** no tenían ninguna anotación real cerca —
  revisados 34 a mano en la app (`data/revision_sin_anotacion.csv`, todos
  confirmados: 30 ruido, 4 alimentación, cero desacuerdo con lo que el cluster
  sugería) y promovidos a `categoria_real` (celda "Promover veredictos
  manuales" de este notebook). Quedan **115 sin revisar todavía**.
- **Servido sigue sin cluster propio** — probablemente porque dura muy poco
  (mediana real 120s) y comparte ese rango de duración con ruido corto.
  Candidato a investigar: separar servido de ruido necesita otra variable
  además de duración (ej. `delta_neto_real` — servido debería ser siempre
  positivo y grande, ruido más disperso en signo y magnitud).
